# Seminário de Ferramentas para Data Science: Altair
**Disciplina:** Tecnologias para Data Science - DCOMP / UFSJ   
**Professora:** Prof.ª Carolina Xavier   
**Dupla:** Bryan e Cassiuscray   
**Conjunto de dados do estudo de caso:** Titanic (`titanic.csv`)

---

## 1. Contextualização e Propósito da Ferramenta

### O problema que o Altair resolve no ciclo de Ciência de Dados
Durante a **Análise Exploratória de Dados (EDA)** e a comunicação de resultados, um gargalo comum é passar de gráficos estáticos básicos para visualizações interativas sem perder clareza nem escrever código excessivamente verboso. Em muitos cenários, o analista quer inspecionar pontos específicos ao passar o mouse (*tooltips*), aplicar zoom/deslocamento ou cruzar informações entre dois gráficos para testar hipóteses rapidamente dentro do próprio *notebook*.

O **Altair** é uma biblioteca em Python de **visualização estatística declarativa**, construída sobre a **Gramática de Gráficos** (*Vega-Lite*), voltada para exploração visual e análise interativa de dados tabulares. Em vez de descrever passo a passo *como* desenhar cada elemento visual na tela, você declara **o que** cada coluna dos seus dados representa visualmente (posição nos eixos `x` e `y`, cor, tamanho, *tooltip*, filtros e interações). A biblioteca gera uma especificação declarativa (JSON no padrão Vega-Lite) que é renderizada no ambiente interativo.

### Como ele se posiciona em relação às ferramentas vistas na disciplina
Ao longo do curso, trabalhamos diferentes ferramentas para etapas distintas do fluxo de dados:
* **Manipulação e processamento tabular / grandes volumes:** `pandas` e `PySpark` (além de práticas com bases como o Titanic e o CadÚnico). O Altair não substitui o `pandas` no tratamento pesado dos dados, mas **conversa diretamente com `DataFrames` do `pandas`**, servindo como camada de visualização exploratória.
* **Visualização estatística tradicional:** `matplotlib` e `seaborn` foram usados na disciplina para construir gráficos como barras, dispersão, *boxplots* e *heatmaps*. Enquanto o `matplotlib` tem natureza mais imperativa/procedural e exige mais configuração manual para interações, o Altair enfatiza a especificação declarativa por canais visuais e a composição/interação nativa entre gráficos.
* **Geovisualização e aplicações web:** `folium` (como no mapeamento eleitoral) e `streamlit` (para dashboards e aplicações web reativas) atendem, respectivamente, a mapas geográficos interativos e interfaces/painéis web completos. O Altair foca na **exploração visual declarativa de dados tabulares** no *notebook* (podendo também compor visões ligadas entre si).

## 2. Instalação, Configuração e Conceitos Principais

### Reprodutibilidade e Ambiente Isolado

O projeto utiliza um ambiente Conda isolado chamado `seminario-altair`, definido pelo arquivo `environment.yml` na raiz do projeto. Esse arquivo especifica as bibliotecas e versões necessárias para reproduzir os exemplos.

No terminal integrado do VS Code, aberto na pasta do projeto, execute:

```bash
conda env create -f environment.yml
conda activate seminario-altair
```

Se o ambiente já estiver criado, basta ativá-lo. Para executar o notebook diretamente no VS Code, abra `seminario-altair-titanic.ipynb`, selecione o ambiente `seminario-altair` em **Select Kernel** e clique em **Run All**.

As principais bibliotecas utilizadas são:

- **Altair:** criação de gráficos estatísticos declarativos e interativos.
- **Pandas:** leitura, preparação e manipulação dos dados.
- **Jupyter/IPython:** execução e documentação das etapas no notebook.


In [1]:
import pandas as pd
import altair as alt
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

# Garante o renderizador padrão no JupyterLab
alt.renderers.enable("default")

# Carrega a base do Titanic (considerando titanic.csv na mesma pasta do notebook)
df = pd.read_csv("titanic.csv")

# Cria uma cópia filtrando registros sem idade ou tarifa para evitar lacunas nos gráficos
df_clean = df.dropna(subset=["Age", "Fare"]).copy()

# Mapeia variáveis numéricas/códigos para rótulos legíveis nas visualizações
df_clean["Sobreviveu_Texto"] = df_clean["Survived"].map({0: "Não", 1: "Sim"})
df_clean["Classe_Texto"] = df_clean["Pclass"].map({
    1: "1ª Classe",
    2: "2ª Classe",
    3: "3ª Classe"
})
df_clean["Sexo_Texto"] = df_clean["Sex"].map({
    "male": "Masculino",
    "female": "Feminino"
})

print(f"Total de registros originais: {len(df)} | Registros prontos para os gráficos: {len(df_clean)}")
df_clean[["Name", "Classe_Texto", "Sexo_Texto", "Age", "Fare", "Sobreviveu_Texto"]].head()

Total de registros originais: 887 | Registros prontos para os gráficos: 887


,Name,Classe_Texto,Sexo_Texto,Age,Fare,Sobreviveu_Texto
0,Mr. Owen Harris Braund,3ª Classe,Masculino,22.0,7.2500,Não
1,Mrs. John Bradley (Florence Briggs Thayer) Cum...,1ª Classe,Feminino,38.0,71.2833,Sim
2,Miss. Laina Heikkinen,3ª Classe,Feminino,26.0,7.9250,Sim
3,Mrs. Jacques Heath (Lily May Peel) Futrelle,1ª Classe,Feminino,35.0,53.1000,Sim
4,Mr. William Henry Allen,3ª Classe,Masculino,35.0,8.0500,Não


## 3. Demonstração Prática com Dados Reais

Para demonstrar a biblioteca em um cenário real, utilizamos o conjunto de dados do **Titanic** (`titanic.csv`), explorando diferentes tipos de variáveis (`:Q` para idade e tarifa, `:N` para categorias como classe, sexo e sobrevivência).

Abaixo, exploramos **quatro recursos** do Altair:
1. **Gráfico de dispersão interativo com `tooltip` customizado e `interactive()`** (zoom e deslocamento direto no gráfico).
2. **Agregações e facetas declarativas** (contagem/agrupamento direto na especificação do gráfico, sem depender de tabelas auxiliares prévias).
3. **Seleção cruzada (*brushing & linking*)** (arrastar uma área em um gráfico para filtrar/destacar dinamicamente outro gráfico conectado).
4. **Composição de gráficos** (combinar visualizações em camadas ou painéis conectados para leitura exploratória mais rica).

### Recurso 1 — Gráfico de dispersão interativo

Este gráfico relaciona a **idade** dos passageiros com a **tarifa paga**, distinguindo por cores quem sobreviveu e quem não sobreviveu. Ao passar o mouse sobre os pontos, os *tooltips* mostram informações do passageiro. O método `interactive()` permite explorar o gráfico com **zoom e deslocamento**.


In [2]:
# Recurso 1: Gráfico de dispersão declarativo com tooltip detalhado e interação (zoom/pan)
dispersao_titanic = (
    alt.Chart(df_clean)
    .mark_circle(size=55, opacity=0.75)
    .encode(
        x=alt.X("Age:Q", title="Idade (anos)"),
        y=alt.Y("Fare:Q", title="Tarifa paga (Fare)"),
        color=alt.Color(
            "Sobreviveu_Texto:N",
            title="Sobreviveu",
            scale=alt.Scale(domain=["Sim", "Não"], range=["#2b6cb0", "#c53030"])
        ),
        tooltip=[
            alt.Tooltip("Name:N", title="Passageiro"),
            alt.Tooltip("Classe_Texto:N", title="Classe"),
            alt.Tooltip("Sexo_Texto:N", title="Sexo"),
            alt.Tooltip("Age:Q", title="Idade"),
            alt.Tooltip("Fare:Q", title="Tarifa", format=".2f"),
            alt.Tooltip("Sobreviveu_Texto:N", title="Sobreviveu")
        ]
    )
    .properties(
        title="Relação entre Idade, Tarifa e Sobrevivência no Titanic",
        width=520,
        height=320
    )
    .interactive()  # Permite zoom com roda do mouse e arraste (pan)
)

dispersao_titanic

alt.Chart(...)

### Recurso 2 — Agregações e facetas

O Altair calcula diretamente a **contagem de passageiros** em cada situação de sobrevivência usando `count()` e separa os resultados por **classe do bilhete** com `column`. Isso permite comparar as classes em painéis distintos sem precisar criar previamente uma tabela de contagens.


In [3]:
# Recurso 2: Agregação declarativa (count()) e facetas por classe social (column)
barras_facetadas = (
    alt.Chart(df_clean)
    .mark_bar()
    .encode(
        x=alt.X("Sobreviveu_Texto:N", title="Sobreviveu"),
        y=alt.Y("count():Q", title="Quantidade de passageiros"),
        color=alt.Color(
            "Sobreviveu_Texto:N",
            title="Sobreviveu",
            scale=alt.Scale(domain=["Sim", "Não"], range=["#2b6cb0", "#c53030"])
        ),
        column=alt.Column(
            "Classe_Texto:N",
            title="Classe do bilhete",
            sort=["1ª Classe", "2ª Classe", "3ª Classe"]
        ),
        tooltip=[
            alt.Tooltip("Classe_Texto:N", title="Classe"),
            alt.Tooltip("Sobreviveu_Texto:N", title="Sobreviveu"),
            alt.Tooltip("count():Q", title="Total de passageiros")
        ]
    )
    .properties(
        width=170,
        height=260,
        title="Passageiros por Sobrevivência e Classe (Agregação e Facetas no Altair)"
    )
)

barras_facetadas

alt.Chart(...)

### Recurso 3 — Seleção cruzada (*brushing & linking*)

Arraste uma caixa sobre os pontos do **gráfico da esquerda** para selecionar passageiros em uma faixa de idade e tarifa. A seleção destaca os pontos correspondentes e atualiza automaticamente as **barras à direita**, que mostram a quantidade de passageiros selecionados por sexo. Os dois gráficos estão conectados por `selection_interval()` e `transform_filter()`.


In [6]:
# Recurso 3: Seleção cruzada (brushing & linking) — arrastar em um gráfico atualiza o outro
selecao_area = alt.selection_interval()

# Painel esquerdo: dispersão onde o usuário arrasta para selecionar um intervalo de idade e tarifa
painel_selecao = (
    alt.Chart(df_clean)
    .mark_circle(size=55)
    .encode(
        x=alt.X("Age:Q", title="Idade (arraste para selecionar)"),
        y=alt.Y("Fare:Q", title="Tarifa (Fare)"),
        color=alt.condition(
            selecao_area,
            alt.Color("Sobreviveu_Texto:N", title="Sobreviveu"),
            alt.value("lightgray")
        ),
        tooltip=[
            alt.Tooltip("Name:N", title="Passageiro"),
            alt.Tooltip("Classe_Texto:N", title="Classe"),
            alt.Tooltip("Sexo_Texto:N", title="Sexo"),
            alt.Tooltip("Age:Q", title="Idade"),
            alt.Tooltip("Fare:Q", title="Tarifa", format=".2f"),
            alt.Tooltip("Sobreviveu_Texto:N", title="Sobreviveu")
        ]
    )
    .add_params(selecao_area)
    .properties(
        width=390,
        height=280,
        title="1. Arraste uma caixa sobre os pontos"
    )
)

# Painel direito: barras que filtram dinamicamente conforme a seleção feita à esquerda
painel_conectado = (
    alt.Chart(df_clean)
    .mark_bar()
    .encode(
        x=alt.X("count():Q", title="Passageiros na seleção"),
        y=alt.Y("Sexo_Texto:N", title="Sexo"),
        color=alt.Color(
            "Sexo_Texto:N",
            title="Sexo",
            scale=alt.Scale(domain=["Masculino", "Feminino"], range=["#4a5568", "#d53f8c"])
        ),
        tooltip=[
            alt.Tooltip("Sexo_Texto:N", title="Sexo"),
            alt.Tooltip("count():Q", title="Total na seleção")
        ]
    )
    .transform_filter(selecao_area)
    .properties(
        width=260,
        height=280,
        title="2. Distribuição por sexo na área selecionada"
    )
)

# Composição horizontal dos dois gráficos conectados
(painel_selecao | painel_conectado).resolve_scale(color="independent")

alt.HConcatChart(...)

### Recurso 4 — Composição de gráficos e seleção por classe

O painel da esquerda combina **pontos de idade e tarifa** com uma linha que indica a **tarifa média geral**; essa sobreposição é feita com `+`. O painel da direita apresenta a **tarifa média por classe**. Ao clicar em uma das barras, a classe correspondente fica destacada nos dois painéis, demonstrando a composição e a seleção interativa com `selection_point()`.


In [5]:
# Recurso 4: Composição de camadas (+), média por classe (mean()) e clique na classe para destacar
selecao_classe = alt.selection_point(fields=["Classe_Texto"])

base_tarifa = (
    alt.Chart(df_clean)
    .mark_circle(size=45)
    .encode(
        x=alt.X("Age:Q", title="Idade (anos)"),
        y=alt.Y("Fare:Q", title="Tarifa paga (Fare)"),
        color=alt.Color(
            "Classe_Texto:N",
            title="Classe",
            sort=["1ª Classe", "2ª Classe", "3ª Classe"]
        ),
        opacity=alt.condition(
            selecao_classe,
            alt.value(0.75),
            alt.value(0.08)
        ),
        tooltip=[
            alt.Tooltip("Name:N", title="Passageiro"),
            alt.Tooltip("Classe_Texto:N", title="Classe"),
            alt.Tooltip("Age:Q", title="Idade"),
            alt.Tooltip("Fare:Q", title="Tarifa", format=".2f")
        ]
    )
)

linha_media_tarifa = (
    alt.Chart(df_clean)
    .mark_rule(color="black", strokeDash=[5, 4], size=2)
    .encode(
        y=alt.Y("mean(Fare):Q"),
        tooltip=[alt.Tooltip("mean(Fare):Q", title="Tarifa média geral", format=".2f")]
    )
)

painel_camadas = (base_tarifa + linha_media_tarifa).properties(
    width=380,
    height=280,
    title="Tarifa vs. Idade com Linha de Média Geral"
)

media_por_classe = (
    alt.Chart(df_clean)
    .mark_bar(cursor="pointer")
    .encode(
        x=alt.X("mean(Fare):Q", title="Tarifa média (Fare)"),
        y=alt.Y(
            "Classe_Texto:N",
            title="Classe",
            sort=["1ª Classe", "2ª Classe", "3ª Classe"]
        ),
        color=alt.Color(
            "Classe_Texto:N",
            legend=None,
            sort=["1ª Classe", "2ª Classe", "3ª Classe"]
        ),
        opacity=alt.condition(selecao_classe, alt.value(1.0), alt.value(0.35)),
        tooltip=[
            alt.Tooltip("Classe_Texto:N", title="Classe"),
            alt.Tooltip("mean(Fare):Q", title="Tarifa média", format=".2f")
        ]
    )
    .add_params(selecao_classe)
    .properties(
        width=240,
        height=280,
        title="Média de Tarifa por Classe"
    )
)

painel_camadas | media_por_classe

alt.HConcatChart(...)

## 4. Vantagens, Limitações, Cenários de Uso e Conclusão

### Vantagens do Altair
* **Sintaxe declarativa e legível:** em vez de escrever sequências imperativas de comandos para desenhar cada detalhe, você declara *o que* cada coluna representa visualmente (`x`, `y`, `color`, `tooltip`, agregações e seleções).
* **Interatividade nativa sem boilerplate pesado:** recursos como *tooltips* customizados, zoom/deslocamento e seleção cruzada (*brushing & linking*) entre gráficos fazem parte do modelo da biblioteca.
* **Composição modular:** combinar camadas (`+`), colocar gráficos lado a lado (`|`) ou em grade/facetas é direto e mantém a coerência visual.
* **Integração natural com `pandas`:** trabalha diretamente sobre `DataFrames` do `pandas`, encaixando-se bem na etapa exploratória do fluxo em Python.

### Limitações e Pontos de Atenção
* **Limite padrão de dados embutidos:** por padrão, o Altair impõe um limite de registros embutidos diretamente na especificação JSON enviada ao navegador para evitar *notebooks* pesados ou travamentos; para bases maiores, é preciso ajustar configurações de dados ou pré-agregar/filtrar previamente.
* **Menor flexibilidade para desenhos altamente customizados:** ilustrações técnicas muito fora da gramática padrão ou controles de interface complexos de aplicação web costumam exigir outras ferramentas (`matplotlib` para ajustes gráficos muito específicos ou `streamlit` para aplicações/painéis web completos).
* **Dependência de renderização no ambiente:** como o resultado é renderizado no *front-end* (Vega-Lite no JupyterLab / navegador), problemas de versão ou renderizador podem gerar avisos ou exigir atenção na configuração do ambiente.

### Quando Usar (e Quando Não Usar)
* **Vale usar:** na **Análise Exploratória de Dados (EDA)** interativa em *notebooks*, quando você quer inspecionar registros no hover, cruzar hipóteses entre gráficos conectados ou apresentar visualizações claras e reprodutíveis.
* **Vale considerar outras ferramentas:** quando o objetivo for um dashboard web completo com múltiplos widgets e fluxos de aplicação (`streamlit`), mapas geográficos dedicados (`folium`) ou gráficos estáticos/customizações fora da gramática declarativa.

### Conclusão
O **Altair** oferece um meio-termo muito produtivo entre simplicidade de escrita e interatividade exploratória. Aplicado à base do **Titanic**, ele permitiu transitar de leituras pontuais por passageiro até agregações por classe e seleções cruzadas com poucas linhas de código declarativo, mantendo o fluxo reprodutível dentro do ambiente isolado do projeto.